# اليوم 2، المعمل 3: Pipeline وصيد التسرب

يبني هذا المعمل كائنًا واحدًا يستقبل الصف الخام ويعيد احتمالًا. كل قيمة يتعلمها imputer أو scaler تأتي من بيانات التدريب فقط.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(ROOT / "src"))
DATA = ROOT / "data" / "raw"
RANDOM_STATE = 42


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.utils import shuffle
from manafeth.data import load_customers, split_customers
from manafeth.features import build_preprocessor, LEAKAGE_COLUMNS

df = load_customers(DATA)
X_train, X_test, y_train, y_test = split_customers(df)


## مهمتك الأولى

كوّن Pipeline من `build_preprocessor()` وLogisticRegression. نفّذ 75/25 validation داخل train.

In [ ]:
from sklearn.model_selection import train_test_split
X_fit, X_valid, y_fit, y_valid = train_test_split(X_train, y_train, test_size=.25, stratify=y_train, random_state=RANDOM_STATE)
pipe = Pipeline([("prep", build_preprocessor()), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
pipe.fit(X_fit, y_fit)
print("honest PR-AUC:", round(average_precision_score(y_valid, pipe.predict_proba(X_valid)[:, 1]), 3))

y_shuffled = shuffle(y_fit, random_state=RANDOM_STATE).to_numpy()
null_pipe = Pipeline([("prep", build_preprocessor()), ("model", LogisticRegression(max_iter=1000))])
null_pipe.fit(X_fit, y_shuffled)
print("shuffled-label PR-AUC:", round(average_precision_score(y_valid, null_pipe.predict_proba(X_valid)[:, 1]), 3))


## اختبار الملصقات العشوائية

إذا خلطنا y يجب أن يهبط PR-AUC قرب 0.14. نتيجة مرتفعة تعني أن مسارًا ما يكشف الإجابة.

In [ ]:
# أُنجزت خطوات هذا القسم في خلية الحل السابقة.


## تحدي التسرب

جرّب مؤقتًا إضافة `next_month_orders` من dataframe قبل التقسيم. لاحظ القفزة، ثم احذف العمود واشرح لماذا لا يمثل إنجازًا.

In [ ]:
# لا تضع العمود المسرّب في النسخة النهائية. اكتب تفسيرك هنا.


**ناتج التسليم:** Pipeline نظيف، نتيجة صحيحة، وشرح زمني لكل عمود مستبعد.